In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from ast import literal_eval
import geopandas as gpd


C:\Users\abreunig\AppData\Local\Temp\2\ipykernel_12904\305143733.py:5: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


In [2]:
CO_URL = 'https://www.mountainproject.com/area/105708956/colorado'

In [3]:
html_str = requests.get(CO_URL).content
soup = BeautifulSoup(html_str, 'html.parser')

In [4]:
co_areas = {}
for link in soup.find_all('div', {'class': 'lef-nav-row'}):
    for url in link.find_all('a', href=True):
        print(url.contents)
        print(url['href'])
        co_areas[url.contents[0]] = url['href']

['10 Mile Canyon']
https://www.mountainproject.com/area/106817352/10-mile-canyon
['Alpine Rock']
https://www.mountainproject.com/area/105744466/alpine-rock
['Aspen Glen Campground Bouldering']
https://www.mountainproject.com/area/122921638/aspen-glen-campground-bouldering
['Black Hawk']
https://www.mountainproject.com/area/121624024/black-hawk
['Boulder']
https://www.mountainproject.com/area/105801420/boulder
['Breckenridge']
https://www.mountainproject.com/area/108289116/breckenridge
['Bridge']
https://www.mountainproject.com/area/124014835/bridge
['Broomfield']
https://www.mountainproject.com/area/122059772/broomfield
['Broomfield Boulders']
https://www.mountainproject.com/area/123696872/broomfield-boulders
['Buena Vista']
https://www.mountainproject.com/area/105744391/buena-vista
['Canon City']
https://www.mountainproject.com/area/105800427/canon-city
['Carbondale Area']
https://www.mountainproject.com/area/105802064/carbondale-area
['Cataract Lake (Heeney, Colorado)']
https://www.m

In [5]:
co_areas

{'10 Mile Canyon': 'https://www.mountainproject.com/area/106817352/10-mile-canyon',
 'Alpine Rock': 'https://www.mountainproject.com/area/105744466/alpine-rock',
 'Aspen Glen Campground Bouldering': 'https://www.mountainproject.com/area/122921638/aspen-glen-campground-bouldering',
 'Black Hawk': 'https://www.mountainproject.com/area/121624024/black-hawk',
 'Boulder': 'https://www.mountainproject.com/area/105801420/boulder',
 'Breckenridge': 'https://www.mountainproject.com/area/108289116/breckenridge',
 'Bridge': 'https://www.mountainproject.com/area/124014835/bridge',
 'Broomfield': 'https://www.mountainproject.com/area/122059772/broomfield',
 'Broomfield Boulders': 'https://www.mountainproject.com/area/123696872/broomfield-boulders',
 'Buena Vista': 'https://www.mountainproject.com/area/105744391/buena-vista',
 'Canon City': 'https://www.mountainproject.com/area/105800427/canon-city',
 'Carbondale Area': 'https://www.mountainproject.com/area/105802064/carbondale-area',
 'Cataract Lak

In [6]:
colorado_areas = pd.DataFrame.from_dict(co_areas, orient='index')
colorado_areas

,0
10 Mile Canyon,https://www.mountainproject.com/area/106817352...
Alpine Rock,https://www.mountainproject.com/area/105744466...
Aspen Glen Campground Bouldering,https://www.mountainproject.com/area/122921638...
Black Hawk,https://www.mountainproject.com/area/121624024...
Boulder,https://www.mountainproject.com/area/105801420...
...,...
Telluride/Norwood area,https://www.mountainproject.com/area/105969849...
Vogel Canyon,https://www.mountainproject.com/area/107841192...
Weston Pass-(Weston Wall),https://www.mountainproject.com/area/106689909...
"Wet Mountains, The",https://www.mountainproject.com/area/105801976...


In [7]:
def get_coords(contents_list):
    coord_str = contents_list[0].strip() # remove linebreak/whitespace
    coord_list = coord_str.split(',')
    lat = float(coord_list[0])
    lng = float(coord_list[1])

    return lat, lng

In [8]:
all_crags = pd.DataFrame()
for area, url in colorado_areas.iterrows():
    print(area)
    crag_coords = {}
    html_str = requests.get(url[0]).content
    area_soup = BeautifulSoup(html_str, 'html.parser')
    
    for link in area_soup.find_all('div', {'class': 'lef-nav-row'}):
        for url in link.find_all('a', href=True):
            crag_name = url.contents[0]

            html_str = requests.get(url['href']).content
            crag = BeautifulSoup(html_str, 'html.parser')
            
            for c in crag.find_all('div', {'class': 'col-md-3 left-nav float-md-left mb-2'}):
                for header in c.find_all('h3'):
                    if 'Areas in' in header.text:
                        print(f'Has subareas: {header.text}')

                        # get subarea names
                        for link in crag.find_all('div', {'class': 'lef-nav-row'}):
                            for url in link.find_all('a', href=True):
                                sub_crag = url.contents[0]
                                print(crag_name)
                                html_str = requests.get(url['href']).content
                                subarea_soup = BeautifulSoup(html_str, 'html.parser')

                                for c in subarea_soup.find_all('div', {'class': 'small'}):
                                    # Grabbing lat + long
                                    for text in c.find_all('td'):
                                        if '-1' in text.text:
                                            contents_list = text.contents
                                            lat, lng = get_coords(contents_list)
                                            #print(lat, lng)
                                            crag_coords = {'Lat': lat, 'Long': lng}
                                            #print(crag_coords)


                                # total climbs stats
                                for c in subarea_soup.find_all('div', {'class': 'col-lg-6 text-xs-center', 'id': 'route-count-container'}):
                                    script_tag = c.find_all('script')[1].string
                                    if script_tag:
                                        tag = script_tag.split(';')[0].replace(' ', '').replace('\n', '')
                                        data = tag.split('google.visualization.arrayToDataTable(')[1].rstrip(')')
                                        data = literal_eval(data)[1:]
                                        climb_dict = {k[0]: k[1] for k in data}
                                        crag_dict = climb_dict | crag_coords
                                        crag_id = pd.MultiIndex.from_tuples([(crag_name, sub_crag)], name=['Area', 'Crag'])
                                        crag_df = pd.DataFrame(crag_dict, index=crag_id)
                                        all_crags = pd.concat([all_crags, crag_df])


                    else:
                        print(f'No subareas: {header.text}')
                        
                        for c in crag.find_all('div', {'class': 'small'}):
                            # Grabbing lat + long
                            for text in c.find_all('td'):
                                if '-1' in text.text:
                                    contents_list = text.contents
                                    lat, lng = get_coords(contents_list)
                                    #print(lat, lng)
                                    crag_coords = {'Lat': lat, 'Long': lng}
                                    #print(crag_coords)


                        # total climbs stats
                        for c in crag.find_all('div', {'class': 'col-lg-6 text-xs-center', 'id': 'route-count-container'}):
                            script_tag = c.find_all('script')[1].string
                            if script_tag:
                                tag = script_tag.split(';')[0].replace(' ', '').replace('\n', '')
                                data = tag.split('google.visualization.arrayToDataTable(')[1].rstrip(')')
                                data = literal_eval(data)[1:]
                                climb_dict = {k[0]: k[1] for k in data}
                                crag_dict = climb_dict | crag_coords
                                crag_id = pd.MultiIndex.from_tuples([(area, crag_name)], name=['Area', 'Crag'])
                                crag_df = pd.DataFrame(crag_dict, index=crag_id)
                                all_crags = pd.concat([all_crags, crag_df])
                                
                                #all_crag_ids.append(crag_tuple)
                    
               

    
                
                
            

    

10 Mile Canyon
No subareas: Routes in 10 Mile Tower
Has subareas: Areas in Brick Wall/Alcoves
Brick Wall/Alcoves
Brick Wall/Alcoves
No subareas: Routes in The Cache Wall
No subareas: Routes in Desperation Wall
Has subareas: Areas in The Diamond /Officer's Wall Massif
Diamond /Officer's Wall Massif, The
Diamond /Officer's Wall Massif, The
No subareas: Routes in The Dome
No subareas: Routes in Halfway Rock
No subareas: Routes in Mount Royal (near Frisco)
No subareas: Routes in Officer's Gulch West
No subareas: Routes in The Shire
Has subareas: Areas in Smile Wall
Smile Wall
Smile Wall
No subareas: Routes in Spider Boulder
No subareas: Routes in Sunshine Buttress (aka Wichita Wall)
Has subareas: Areas in Sunshine, Ding Dong Dome Bouldering
Sunshine, Ding Dong Dome Bouldering
No subareas: Routes in Teeter Top
No subareas: Routes in Wheeler Lakes
No subareas: Routes in White Cliff
Alpine Rock
No subareas: Routes in Capitol Peak
No subareas: Routes in Chimneys of Treasure
No subareas: Routes

In [14]:
all_crags = all_crags.fillna(0)
all_crags.head()

Sport       Lat       Long  Trad  \
Area               Crag                                                 
10 Mile Canyon     10 Mile Tower       1.0  39.57010 -106.11985   0.0   
Brick Wall/Alcoves Alcoves, The        0.0  39.56660 -106.12615   0.0   
                   Brick Wall          0.0  39.56637 -106.12630   1.0   
10 Mile Canyon     Cache Wall, The     6.0  39.56147 -106.13202   1.0   
                   Desperation Wall    4.0  39.51999 -106.14030   2.0   

                                     Toprope  Boulder  Alpine  Aid  Ice  \
Area               Crag                                                   
10 Mile Canyon     10 Mile Tower         0.0      0.0     0.0  0.0  0.0   
Brick Wall/Alcoves Alcoves, The          0.0      0.0     0.0  0.0  0.0   
                   Brick Wall            1.0      0.0     0.0  0.0  0.0   
10 Mile Canyon     Cache Wall, The       0.0      0.0     0.0  0.0  0.0   
                   Desperation Wall      3.0      3.0     0.0  0.0  0.0   

                                     Mixed  Total Climbs  \
Area               Crag                                    
10 Mile Canyon     10 Mile Tower       0.0           1.0   
Brick Wall/Alcoves Alcoves, The        0.0           0.0   
                   Brick Wall          0.0           2.0   
10 Mile Canyon     Cache Wall, The     0.0           7.0   
                   Desperation Wall    0.0          12.0   

                                                        geometry  
Area               Crag                                           
10 Mile Canyon     10 Mile Tower     POINT (-106.11985 39.57010)  
Brick Wall/Alcoves Alcoves, The      POINT (-106.12615 39.56660)  
                   Brick Wall        POINT (-106.12630 39.56637)  
10 Mile Canyon     Cache Wall, The   POINT (-106.13202 39.56147)  
                   Desperation Wall  POINT (-106.14030 39.51999)

In [15]:
all_crags['Total Climbs'] = all_crags.apply(lambda x: x['Sport'] + x['Trad'] + x['Toprope'] + x['Boulder'] + x['Alpine'] + x['Ice'] + x['Aid'] + x['Mixed'], axis=1)

In [16]:
all_crags.to_csv('all_crags_colorado_full.csv')

In [17]:
gdf = gpd.GeoDataFrame(all_crags, geometry=gpd.points_from_xy(all_crags['Long'], all_crags['Lat']), crs=4326)
gdf.head()

Sport       Lat       Long  Trad  \
Area               Crag                                                 
10 Mile Canyon     10 Mile Tower       1.0  39.57010 -106.11985   0.0   
Brick Wall/Alcoves Alcoves, The        0.0  39.56660 -106.12615   0.0   
                   Brick Wall          0.0  39.56637 -106.12630   1.0   
10 Mile Canyon     Cache Wall, The     6.0  39.56147 -106.13202   1.0   
                   Desperation Wall    4.0  39.51999 -106.14030   2.0   

                                     Toprope  Boulder  Alpine  Aid  Ice  \
Area               Crag                                                   
10 Mile Canyon     10 Mile Tower         0.0      0.0     0.0  0.0  0.0   
Brick Wall/Alcoves Alcoves, The          0.0      0.0     0.0  0.0  0.0   
                   Brick Wall            1.0      0.0     0.0  0.0  0.0   
10 Mile Canyon     Cache Wall, The       0.0      0.0     0.0  0.0  0.0   
                   Desperation Wall      3.0      3.0     0.0  0.0  0.0   

                                     Mixed  Total Climbs  \
Area               Crag                                    
10 Mile Canyon     10 Mile Tower       0.0           1.0   
Brick Wall/Alcoves Alcoves, The        0.0           0.0   
                   Brick Wall          0.0           2.0   
10 Mile Canyon     Cache Wall, The     0.0           7.0   
                   Desperation Wall    0.0          12.0   

                                                        geometry  
Area               Crag                                           
10 Mile Canyon     10 Mile Tower     POINT (-106.11985 39.57010)  
Brick Wall/Alcoves Alcoves, The      POINT (-106.12615 39.56660)  
                   Brick Wall        POINT (-106.12630 39.56637)  
10 Mile Canyon     Cache Wall, The   POINT (-106.13202 39.56147)  
                   Desperation Wall  POINT (-106.14030 39.51999)

In [18]:
gdf = gdf.to_crs(epsg=3857)
gdf.head()

Sport       Lat       Long  Trad  \
Area               Crag                                                 
10 Mile Canyon     10 Mile Tower       1.0  39.57010 -106.11985   0.0   
Brick Wall/Alcoves Alcoves, The        0.0  39.56660 -106.12615   0.0   
                   Brick Wall          0.0  39.56637 -106.12630   1.0   
10 Mile Canyon     Cache Wall, The     6.0  39.56147 -106.13202   1.0   
                   Desperation Wall    4.0  39.51999 -106.14030   2.0   

                                     Toprope  Boulder  Alpine  Aid  Ice  \
Area               Crag                                                   
10 Mile Canyon     10 Mile Tower         0.0      0.0     0.0  0.0  0.0   
Brick Wall/Alcoves Alcoves, The          0.0      0.0     0.0  0.0  0.0   
                   Brick Wall            1.0      0.0     0.0  0.0  0.0   
10 Mile Canyon     Cache Wall, The       0.0      0.0     0.0  0.0  0.0   
                   Desperation Wall      3.0      3.0     0.0  0.0  0.0   

                                     Mixed  Total Climbs  \
Area               Crag                                    
10 Mile Canyon     10 Mile Tower       0.0           1.0   
Brick Wall/Alcoves Alcoves, The        0.0           0.0   
                   Brick Wall          0.0           2.0   
10 Mile Canyon     Cache Wall, The     0.0           7.0   
                   Desperation Wall    0.0          12.0   

                                                              geometry  
Area               Crag                                                 
10 Mile Canyon     10 Mile Tower     POINT (-11813207.665 4803665.639)  
Brick Wall/Alcoves Alcoves, The      POINT (-11813908.978 4803160.209)  
                   Brick Wall        POINT (-11813925.676 4803126.996)  
10 Mile Canyon     Cache Wall, The   POINT (-11814562.423 4802419.439)  
                   Desperation Wall  POINT (-11815484.149 4796431.759)

In [19]:
gdf.shape

(2659, 12)

In [20]:
gdf_buffer = gpd.GeoDataFrame(gdf.drop('geometry', axis=1), geometry=gdf.buffer(100, cap_style=3), crs=3857)
#gdf_buffer = gdf_buffer.to_crs(3857)

In [26]:
gdf_buffer.to_file('all_crags_3857_10m.geojson', driver='GeoJSON')

In [21]:
gdf_buffer.to_file('all_crags_final_3857_100m.shp', driver='ESRI Shapefile')

C:\Users\abreunig\AppData\Local\Temp\2\ipykernel_12904\311461025.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_buffer.to_file('all_crags_final_3857_100m.shp', driver='ESRI Shapefile')


In [23]:
all_crags = pd.DataFrame(all_crags, index=index)
all_crags

C:\Users\abreu\AppData\Local\Temp\ipykernel_7480\2170055293.py:1: FutureWarning: reindexing with a non-unique Index is deprecated and will raise in a future version.
  all_crags = pd.DataFrame(all_crags, index=index)


ValueError: cannot reindex on an axis with duplicate labels

In [10]:
all_crags = all_crags.reset_index().rename(columns={'index': 'Crag'})

In [11]:
all_crags

,Crag,Sport,Lat,Long,Trad,Toprope,Boulder
0,10 Mile Tower,1.0,39.57010,-106.11985,NaN,NaN,NaN
1,Brick Wall/Alcoves,NaN,39.56649,-106.12626,1.0,1.0,NaN
2,"Cache Wall, The",6.0,39.56147,-106.13202,1.0,NaN,NaN
3,Desperation Wall,4.0,39.51999,-106.14030,2.0,3.0,3.0
4,"Diamond /Officer's Wall Massif, The",8.0,39.53802,-106.13790,1.0,NaN,1.0
5,"Dome, The",4.0,39.57073,-106.11763,12.0,NaN,NaN
6,Halfway Rock,9.0,39.53880,-106.13640,NaN,NaN,NaN
7,Mount Royal (near Frisco),2.0,39.57230,-106.11100,3.0,NaN,NaN
8,Officer's Gulch West,18.0,39.52984,-106.14082,NaN,1.0,NaN
9,"Shire, The",3.0,39.57229,-106.11700,NaN,1.0,NaN


In [ ]:
index = pd.MultiIndex